In [5]:
import os

from rag.vector_store import MilvusVectorStore

In [7]:
os.environ["OPENAI_API_KEY"] = "sk-ZwGM6SW5SaUkjLn219uF8Jcb22H4rKipOwpqDTwMeYOvBUu8"           # 换成你的 key
os.environ["OPENAI_BASE_URL"] = "https://api.openai-proxy.org/v1"
os.environ["MILVUS_URL"] = "http://192.168.150.102:19530"

# 假设你的 config 有 MILVUS_COLLECTION，没有的话直接设环境变量
os.environ["MILVUS_COLLECTION"] = "test_hybrid"

from rag.models import DocumentChunk, ChunkMetadata, SearchRequest, MetadataFilter

store = MilvusVectorStore(collection_name="my_rag_collection")

# ── 插入几条测试数据 ──
chunks = [
    DocumentChunk(
        content="Milvus 是一个高性能向量数据库",
        metadata=ChunkMetadata(
            doc_id=1, doc_type="guide", chunk_type="text",
            category_id=10, source="milvus_intro.md",
            title="Milvus 简介", chunk_index=0,
        ),
    ),
    DocumentChunk(
        content="LangChain 提供了统一的 Embedding 接口",
        metadata=ChunkMetadata(
            doc_id=2, doc_type="faq", chunk_type="text",
            category_id=20, source="langchain.md",
            title="LangChain Embedding", chunk_index=0,
        ),
    ),
]

n = store.upsert_chunks(chunks)
print(f"插入了 {n} 条")

# ── 搜索 ──
req = SearchRequest(query="向量数据库", top_k=3)
results = store.search(req)
for r in results:
    print(f"  [{r.score:.4f}] {r.metadata.title}: {r.content[:40]}")

# ── 按 doc_id 删除 ──
deleted = store.delete_by_doc_id(1)
print(f"删除了 {deleted} 条")

# ── 统计 ──
print(store.stats())

--> 检测到 embedding 维度: 1536


G:\dev\welove-shop-agt\backend\ai-service\rag\vector_store.py:167: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  meta_list = [self._prepare_metadata(c) for c in chunks]
G:\dev\welove-shop-agt\backend\ai-service\rag\vector_store.py:184: PyMilvusDeprecationWarning: `Collection.insert` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  dense_vectors,
G:\dev\welove-shop-agt\backend\ai-service\rag\vector_store.py:196: PyMilvusDeprecationWarning: `Collection.flush` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  return len(chunks)


插入了 2 条


G:\dev\welove-shop-agt\backend\ai-service\rag\vector_store.py:201: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.load()
G:\dev\welove-shop-agt\backend\ai-service\rag\vector_store.py:202: PyMilvusDeprecationWarning: `Collection.load` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  
G:\dev\welove-shop-agt\backend\ai-service\rag\vector_store.py:210: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  [query_vec],


  [0.6777] Milvus 简介: Milvus 是一个高性能向量数据库
  [0.3111] LangChain Embedding: LangChain 提供了统一的 Embedding 接口
  [0.3111] LangChain Embedding: LangChain 提供了统一的 Embedding 接口
删除了 1 条
{'provider': 'milvus', 'collection': 'my_rag_collection'}
